In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from sklearn.metrics import accuracy_score

In [ ]:
class NN:
    def __init__(self, input_size=784, hidden_nodes=128, output_size=10):
        self.weights1 = np.random.randn(input_size, hidden_nodes) * np.sqrt(2 / input_size)
        self.weights2 = np.random.randn(hidden_nodes, output_size) * np.sqrt(2 / hidden_nodes)
        self.bias1 = np.zeros((1, hidden_nodes))
        self.bias2 = np.zeros((1, output_size))

    def relu(self, z):
        return np.maximum(0, z)

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, x):
        self.x = x
        self.z1 = x @ self.weights1 + self.bias1
        self.a1 = self.relu(self.z1)
        self.z2 = self.a1 @ self.weights2 + self.bias2
        self.a2 = self.softmax(self.z2)
        return self.a2

    def cross_entropy(self, pred, y):
        epsilon = 1e-10
        correct_probs = pred[np.arange(len(y)), y]
        loss = -np.mean(np.log(correct_probs + epsilon))
        return loss

    def accuracy(self, pred, y):
        predictions = np.argmax(pred, axis=1)
        return np.mean(predictions == y)
    
    def back(self, y, lr=0.01):
        m = self.x.shape[0]
        dZ2 = self.a2.copy()
        dZ2[np.arange(m), y] -= 1
        dZ2 /= m
        dW2 = self.a1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ self.weights2.T
        dZ1 = dA1 * (self.z1 > 0)
        dW1 = self.x.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        self.weights1 -= lr * dW1
        self.bias1 -= lr * db1
        self.weights2 -= lr * dW2
        self.bias2 -= lr * db2


    def train(self, x, y, epochs=100, lr=0.01):
        batch_size = 64
        for epochs in range(epochs):
            indices = np.random.permutation(len(x))
            x_shuffled = x[indices]
            y_shuffled = y[indices]
            for start in range(0, len(x), batch_size):
                end = start + batch_size
                xb = x_shuffled[start:end]
                yb = y_shuffled[start:end]
                self.pred = self.forward(xb)
                self.back(yb, lr)

    def predict(self, x):
        self.pred = self.forward(x)
        return np.argmax(self.pred, axis=1)

In [ ]:
(x_train,y_train),(x_test,y_test)=keras.datasets.mnist.load_data()

In [ ]:
x_train=x_train/255.0
x_test=x_test/255.0
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)

In [ ]:
print(x_train.shape,y_train.shape,x_test.shape,y_test.shape)

(60000, 784) (60000,) (10000, 784) (10000,)


In [ ]:
model=NN(hidden_nodes=258)
model.train(x_train,y_train,epochs=100,lr=0.1)

Epoch 1/100 | Loss: 2.4255 | Accuracy: 0.0775
Epoch 2/100 | Loss: 2.2543 | Accuracy: 0.1434
Epoch 3/100 | Loss: 2.1353 | Accuracy: 0.2520
Epoch 4/100 | Loss: 2.0363 | Accuracy: 0.3658
Epoch 5/100 | Loss: 1.9476 | Accuracy: 0.4520
Epoch 6/100 | Loss: 1.8655 | Accuracy: 0.5186
Epoch 7/100 | Loss: 1.7882 | Accuracy: 0.5710
Epoch 8/100 | Loss: 1.7149 | Accuracy: 0.6128
Epoch 9/100 | Loss: 1.6452 | Accuracy: 0.6465
Epoch 10/100 | Loss: 1.5790 | Accuracy: 0.6711
Epoch 11/100 | Loss: 1.5162 | Accuracy: 0.6917
Epoch 12/100 | Loss: 1.4568 | Accuracy: 0.7083
Epoch 13/100 | Loss: 1.4008 | Accuracy: 0.7236
Epoch 14/100 | Loss: 1.3481 | Accuracy: 0.7361
Epoch 15/100 | Loss: 1.2986 | Accuracy: 0.7470
Epoch 16/100 | Loss: 1.2522 | Accuracy: 0.7571
Epoch 17/100 | Loss: 1.2087 | Accuracy: 0.7662
Epoch 18/100 | Loss: 1.1682 | Accuracy: 0.7743
Epoch 19/100 | Loss: 1.1302 | Accuracy: 0.7815
Epoch 20/100 | Loss: 1.0948 | Accuracy: 0.7885
Epoch 21/100 | Loss: 1.0617 | Accuracy: 0.7946
Epoch 22/100 | Loss: 1

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
accuracy_score(y_test,y_pred)

0.89